# OHLCV2 bars and the `agg=dict` form

When you have a stream of trades -- each described by a price and a signed
volume (positive = buy, negative = sell) -- you can compress them into
OHLCV bars with separate buy and sell volume in one call:

```python
bars = resample(np.column_stack([price, signed_vol]), t, every=BAR, agg="ohlcv2")
```

The `agg="ohlcv2"` shortcut is convenient, but it is a fixed schema. This
notebook teaches the flexible `agg=dict` form, which lets you define your
own bar columns. The teaching arc is:

1. Use `"ohlcv2"` to get bars and plot them.
2. Rebuild the same bars with a dict to see what the shortcut expands to.
3. Extend the dict with custom per-bar statistics that `"ohlcv2"` cannot produce.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from screamer import PosPart, NegPart, ExpandingStd, ExpandingSkew
from screamer.streams import resample

rng = np.random.default_rng(7)
n   = 400
BAR = 40

price      = 100 + np.cumsum(rng.normal(size=n))
signed_vol = rng.normal(size=n) * rng.integers(1, 5, size=n)  # positive=buy, negative=sell
t          = np.arange(n, dtype=np.int64)

## The `ohlcv2` shortcut

`agg="ohlcv2"` accepts a two-column input `[price, signed_vol]` and
produces six named columns: `open`, `high`, `low`, `close`, `buy_vol`,
`sell_vol`. Access them by name or via `.values` for the raw 2-D array.

In [ ]:
bars = resample(np.column_stack([price, signed_vol]), t, every=BAR, agg="ohlcv2")

print("columns :", bars.columns)
print("shape   :", bars.values.shape)
print()
print(f"{'bar':>4}  {'open':>8}  {'high':>8}  {'low':>8}  {'close':>8}  {'buy_vol':>8}  {'sell_vol':>9}")
for i in range(len(bars.values)):
    print(f"{i:>4}  {bars['open'][i]:>8.2f}  {bars['high'][i]:>8.2f}  {bars['low'][i]:>8.2f}  {bars['close'][i]:>8.2f}  {bars['buy_vol'][i]:>8.2f}  {bars['sell_vol'][i]:>9.2f}")

## Price + volume chart

A two-pane figure: candlestick price chart on top, buy/sell volume bars
below. Green = bullish bar (close >= open); red = bearish. Buy volume
extends upward, sell volume downward.

In [ ]:
fig = plt.figure(figsize=(10, 6))
gs  = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.08)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

x = np.arange(len(bars.values))
o = bars["open"]
h = bars["high"]
l = bars["low"]
c = bars["close"]

for i in range(len(x)):
    col = "green" if c[i] >= o[i] else "red"
    ax1.plot([x[i], x[i]], [l[i], h[i]], color=col, lw=1)   # wick
    ax1.plot([x[i], x[i]], [o[i], c[i]], color=col, lw=6)   # body

ax1.set_ylabel("price")
ax1.set_title("OHLCV2 bars from trades")
plt.setp(ax1.get_xticklabels(), visible=False)

ax2.bar(x,  bars["buy_vol"],  color="green", width=0.6, label="buy")
ax2.bar(x, -bars["sell_vol"], color="red",   width=0.6, label="sell")
ax2.axhline(0, color="k", lw=0.5)
ax2.set_ylabel("volume")
ax2.set_xlabel("bar")
ax2.legend(loc="upper right", fontsize=8)

plt.show()

## Build the same bars with a dict

`agg="ohlcv2"` is sugar for two separate operations:

- **OHLC** on the price column -- equivalent to `{"open":"first", "high":"max", "low":"min", "close":"last"}`.
- **Buy volume** -- sum of `PosPart(signed_vol)` (non-negative ticks only).
- **Sell volume** -- sum of `NegPart(signed_vol)` (magnitude of negative ticks).

Rebuilding them separately confirms the equivalence and shows how each
piece is constructed.

In [ ]:
# OHLC via dict -- identical to the first four columns of ohlcv2
ohlc_dict = resample(
    price, t, every=BAR,
    agg={"open": "first", "high": "max", "low": "min", "close": "last"},
)

# Buy and sell volume via PosPart / NegPart + sum
buy  = resample(np.asarray(PosPart()(signed_vol)), t, every=BAR, agg="sum")
sell = resample(np.asarray(NegPart()(signed_vol)), t, every=BAR, agg="sum")

# Verify they match ohlcv2
ohlc_ok = np.allclose(ohlc_dict.values, bars.values[:, :4])
buy_ok   = np.allclose(buy.values, bars["buy_vol"])
sell_ok  = np.allclose(sell.values, bars["sell_vol"])

print("dict OHLC  == ohlcv2 OHLC columns :", ohlc_ok)
print("PosPart sum == buy_vol             :", buy_ok)
print("NegPart sum == sell_vol            :", sell_ok)

assert ohlc_ok and buy_ok and sell_ok, "Mismatch -- something changed in the API"

## Why the dict form -- extend it

The `"ohlcv2"` shortcut is fixed: you get exactly those six columns. The
dict form lets you choose your own schema. Any screamer functor can be a
reducer: it receives the per-bar tick sequence and returns a scalar.

Here we add `volatility` (expanding standard deviation of price ticks per
bar) and `skew` (expanding skewness) alongside the standard OHLC columns.

In [ ]:
rich = resample(
    price, t, every=BAR,
    agg={
        "open":       "first",
        "high":       "max",
        "low":        "min",
        "close":      "last",
        "volatility": ExpandingStd(),
        "skew":       ExpandingSkew(),
    },
)

print("custom bar columns:", rich.columns)
print()
print(f"{'bar':>4}  {'volatility':>10}  {'skew':>8}")
for i in range(len(rich.values)):
    print(f"{i:>4}  {rich['volatility'][i]:>10.4f}  {rich['skew'][i]:>8.4f}")

fig2, ax = plt.subplots(figsize=(8, 3))
ax.bar(np.arange(len(rich.values)), rich["volatility"], width=0.6, color="steelblue")
ax.set_xlabel("bar")
ax.set_ylabel("within-bar volatility")
ax.set_title("Per-bar price volatility (ExpandingStd over bar ticks)")
plt.tight_layout()
plt.show()

The dict form is how you define your own bar schema. Built-in string
shortcuts (`"ohlcv2"`, `"ohlc"`, `"mean"`, ...) are convenient starting
points; `agg=dict` is the escape hatch when you need anything beyond them.